# v28 — четыре направления улучшения в одном запуске

**Restart Kernel → Run All** в прежнем research `.venv`. 57 конфигураций: контроль v25, состав ансамбля, power normalization, gallery centering, flip-TTA. Все направления включены; лимита времени нет.

Активный MVP, веса, bbox, validation и порог не меняются. Обучения и интернета нет. TTA требует нового CPU-инференса исходных изображений. Выбор только на calibration; validation — контроль и один общий победитель. Это адаптивный development-поиск, не независимый тест.

In [1]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/multi_hypothesis.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import Markdown, display
from collections import Counter
from training import multi_hypothesis as experiment
RUN_NAME = 'multi_hypothesis_v1'
SOURCE_RUN = 'late_fusion_v1'
ALLOW_OUTER_EVALUATION = True
print('Kernel:', sys.executable, '\nRun:', RUN_NAME)
print('Конфигурации:', dict(Counter(s['family'] for s in experiment.systems())))
print('CPU; без обучения и лимита времени; TTA сохраняет каждые 128 изображений')

Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Run: multi_hypothesis_v1
Конфигурации: {'control': 1, 'members': 24, 'power': 6, 'center': 8, 'flip': 18}
CPU; без обучения и лимита времени; TTA сохраняет каждые 128 изображений


## 1. Проверки компонентов

Синтетические проверки состава, формул, независимости query, одинаковых кандидатов, перезапуска после ошибок и сохранения TTA. Новые реальные гипотезы здесь ещё не оцениваются.

In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_multi_hypothesis.py', 'tests/test_late_fusion.py', 'tests/test_map_search.py', 'tests/test_dual_role_inference.py'], check=True)

...........................................................              [100%]
59 passed in 8.81s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_multi_hypothesis.py', 'tests/test_late_fusion.py', 'tests/test_map_search.py', 'tests/test_dual_role_inference.py'], returncode=0)

## 2. Проверка исходных файлов и фиксация плана

Не редактируй код между ячейками. После прерывания используй то же RUN_NAME; после изменения кода требуется новое имя запуска.

In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Results:', context['output'])
display(context['manifest']['multi_hypothesis_plan'])

Preflight OK: 57 configurations, four families, no training or promotion
Results: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_28_multi_hypothesis/runs/multi_hypothesis_v1


{'source_directory': '/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_27_late_fusion/runs/late_fusion_v1',
 'source_signature': 'efd73589b36c79ff48688cfbdaa9fa33bb8c8c0dbfd3f5d21d118af43a364471',
 'families': ['members', 'power', 'center', 'flip'],
 'systems': [{'name': 'V25_control', 'family': 'control', 'lambda': 0.5},
  {'name': 'members_0_w50_l50',
   'family': 'members',
   'members': [0],
   'r1_weight': 0.5,
   'lambda': 0.5},
  {'name': 'members_0_w50_l75',
   'family': 'members',
   'members': [0],
   'r1_weight': 0.5,
   'lambda': 0.75},
  {'name': 'members_0_w75_l50',
   'family': 'members',
   'members': [0],
   'r1_weight': 0.75,
   'lambda': 0.5},
  {'name': 'members_0_w75_l75',
   'family': 'members',
   'members': [0],
   'r1_weight': 0.75,
   'lambda': 0.75},
  {'name': 'members_1_w50_l50',
   'family': 'members',
   'members': [1],
   'r1_weight': 0.5,
   'lambda': 0.5},
  {'name': 'members_1_w50_l75',
 

## 3. Все направления → общий выбор → validation

Первые 39 конфигураций используют готовые признаки. Затем извлекаются flip-признаки calibration и выполняются 18 TTA-сравнений. Для validation flip считается, только если TTA стал общим победителем.

Изолированная ошибка записывается, независимые опыты продолжаются. При неполном наборе выбор не фиксируется. Повреждённые исходники или кэш останавливают запуск. Нет автоматического переключения MVP.

In [4]:
results = experiment.run(context, allow_outer=ALLOW_OUTER_EVALUATION)
print('Status:', results['status'], '| promoted:', results['promoted'])


CALIBRATION 1/57 | control | elapsed 10.4s
[START] calibration_V25_control | elapsed 0.00 h
calibration V25_control: mAP=0.826410; candidates unchanged
[DONE] calibration_V25_control | 0.0 min

CALIBRATION 2/57 | members | elapsed 10.7s
[START] calibration_members_0_w50_l50 | elapsed 0.00 h
calibration members_0_w50_l50: mAP=0.813485; candidates unchanged
[DONE] calibration_members_0_w50_l50 | 0.0 min

CALIBRATION 3/57 | members | elapsed 11.0s
[START] calibration_members_0_w50_l75 | elapsed 0.00 h
calibration members_0_w50_l75: mAP=0.819646; candidates unchanged
[DONE] calibration_members_0_w50_l75 | 0.0 min

CALIBRATION 4/57 | members | elapsed 11.3s
[START] calibration_members_0_w75_l50 | elapsed 0.00 h
calibration members_0_w75_l50: mAP=0.803659; candidates unchanged
[DONE] calibration_members_0_w75_l50 | 0.0 min

CALIBRATION 5/57 | members | elapsed 11.6s
[START] calibration_members_0_w75_l75 | elapsed 0.00 h
calibration members_0_w75_l75: mAP=0.812731; candidates unchanged
[DONE

## 4. Общий отчёт

Если предыдущая ячейка завершилась ошибкой, отчёт всё равно сохранён — эту ячейку можно выполнить отдельно. Готовые задачи не будут повторяться при следующем Run All.

In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
print('Reports, task errors and exports:', context['output'])

# v28 — четыре направления в одном Run All

Статус: complete
57 конфигураций: v25 + 24 состава ансамбля + 6 power + 8 center + 18 flip-TTA.
Это адаптивный development-поиск, не независимый hidden test. Никакого автоматического продвижения.

## Calibration по направлениям

| Семейство | Готово / всего | Лучший вариант | mAP@10 |
|---|---:|---|---:|
| control | 1/1 | V25_control | 0.826409928470033 |
| members | 24/24 | members_12_w75_l50 | 0.8408194759385236 |
| power | 6/6 | power_50_l50 | 0.8333876951864757 |
| center | 8/8 | center_100_l50 | 0.8257143932550437 |
| flip | 18/18 | flip_mvp_w75_l50 | 0.8287287069299264 |

## Validation: контроль и один общий победитель

| Вариант | mAP@10 | Rank-1 | F1 | TNR |
|---|---:|---:|---:|---:|
| V25_control | 0.828957 | 0.829960 | 0.795455 | 0.709677 |
| members_12_w75_l50 | 0.809605 | 0.805668 | 0.795455 | 0.709677 |

ΔmAP: -0.019352; AP лучше/хуже/равно: 16/25/206.

## Все calibration-варианты

| Вариант | mAP@10 |
|---|---:|
| V25_control | 0.826409928470033 |
| members_0_w50_l50 | 0.8134848044909021 |
| members_0_w50_l75 | 0.8196461985878363 |
| members_0_w75_l50 | 0.8036591818299136 |
| members_0_w75_l75 | 0.8127309514591745 |
| members_1_w50_l50 | 0.8192934572202865 |
| members_1_w50_l75 | 0.8189602959521659 |
| members_1_w75_l50 | 0.8208918086204672 |
| members_1_w75_l75 | 0.8201208758119327 |
| members_2_w50_l50 | 0.8130070546737214 |
| members_2_w50_l75 | 0.810662136128164 |
| members_2_w75_l50 | 0.8184658235471243 |
| members_2_w75_l75 | 0.8165882156837442 |
| members_01_w50_l50 | 0.8204968383017164 |
| members_01_w50_l75 | 0.8186885942272121 |
| members_01_w75_l50 | 0.8266919925151632 |
| members_01_w75_l75 | 0.826286510087323 |
| members_02_w50_l50 | 0.8183509700176367 |
| members_02_w50_l75 | 0.8193047566813536 |
| members_02_w75_l50 | 0.8144771906052394 |
| members_02_w75_l75 | 0.817832623564331 |
| members_12_w50_l50 | 0.8262430636211123 |
| members_12_w50_l75 | 0.8208236008947389 |
| members_12_w75_l50 | 0.8408194759385236 |
| members_12_w75_l75 | 0.8331541704305931 |
| power_50_l50 | 0.8333876951864757 |
| power_50_l75 | 0.8261766111141837 |
| power_75_l50 | 0.8263855013550135 |
| power_75_l75 | 0.8209980320041296 |
| power_125_l50 | 0.8232088625874922 |
| power_125_l75 | 0.8236940788058673 |
| center_25_l50 | 0.8256712156407278 |
| center_25_l75 | 0.8210948186862821 |
| center_50_l50 | 0.8253411192842087 |
| center_50_l75 | 0.8232967157052521 |
| center_75_l50 | 0.8252365896674839 |
| center_75_l75 | 0.8209915795586528 |
| center_100_l50 | 0.8257143932550437 |
| center_100_l75 | 0.820723803071364 |
| flip_mvp_w25_l50 | 0.8213187415887765 |
| flip_mvp_w25_l75 | 0.8205734073213748 |
| flip_mvp_w50_l50 | 0.8245702133608638 |
| flip_mvp_w50_l75 | 0.8223789628769304 |
| flip_mvp_w75_l50 | 0.8287287069299264 |
| flip_mvp_w75_l75 | 0.825185615348217 |
| flip_r1_w25_l50 | 0.8262687120918828 |
| flip_r1_w25_l75 | 0.8215176689465307 |
| flip_r1_w50_l50 | 0.8230102885779425 |
| flip_r1_w50_l75 | 0.8210516948423452 |
| flip_r1_w75_l50 | 0.8236996248363845 |
| flip_r1_w75_l75 | 0.8213698541747322 |
| flip_both_w25_l50 | 0.8232207535227279 |
| flip_both_w25_l75 | 0.8245489202907902 |
| flip_both_w50_l50 | 0.8270054738245796 |
| flip_both_w50_l75 | 0.8272852411063792 |
| flip_both_w75_l50 | 0.8254563568878319 |
| flip_both_w75_l75 | 0.8230304985589537 |

## Проверки и ограничения

Порог, raw R1 кандидат и confidence, исходная validation, bbox и evaluator неизменны.
Center использует только среднее статической gallery. Другие query недоступны всем scorer.
Query expansion не реализован (организаторы, вопрос 10). Flip-TTA — того же изображения, без правки bbox.
Экспорт: реальные 2048 признаков; для flip-победителя 4096 = original2048 + flipped2048. NPY replay проверяется.
Стоимость TTA выше: нужны дополнительные encoder passes; официальный performance здесь не измеряется.
Изолированная ошибка не прекращает независимые опыты, но неполный набор блокирует отбор/validation.
После исправления причины повторный Run All продолжает тот же RUN_NAME; код во время серии менять нельзя.
[Источник идеи power normalization](https://europe.naverlabs.com/wp-content/uploads/2010/09/PSM10_0766.pdf): перенос на OSNet и gamma=1.25 — наши гипотезы, не результат статьи.
Источники неизменны: True. Время текущего вызова: 2.5 мин; не скорость инференса.


Reports, task errors and exports: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_28_multi_hypothesis/runs/multi_hypothesis_v1
